
## Power Transformers

A **power transformer** is a mathematical transformation applied to a feature to:

* Reduce **skewness**
* Stabilize **variance**
* Make the distribution more **Gaussian-like (normal)**
* Improve performance of **models that assume normality or linearity**

Instead of just taking a log, power transforms **search for the best exponent** to apply to the data.

So instead of:

$$
x \rightarrow log(x)
$$

we do something like:

$$
x \rightarrow x^{\lambda}
$$

where **λ (lambda)** is learned from the data.

Think of it like a **smart log transform** that adapts to the dataset.


## 1. Box Cox Transform

This is the classic power transformation.
For a value (x):

If λ ≠ 0:
$$
y = \frac{x^{\lambda} - 1}{\lambda}
$$
If λ = 0:
$$
y = log(x)
$$

So **log transform is just a special case of Box-Cox**.

In Box Cox transform, the exponent is a variance called ${\lambda}$ that varies over the range of -5 to 5, and in the process of the searching, we examine all the values of ${\lambda}$.

Finally we choose the optimal value (resuling in the best approximation to a normal distribution) for your variance




In [64]:
# example implementation of box-cox

from sklearn.preprocessing import PowerTransformer
import numpy as np
X = np.array([[1], [10], [100], [1000]])
pt = PowerTransformer(method="box-cox")

X_transformed = pt.fit_transform(X)

### Internal Working of Box–Cox

The Box–Cox transformation finds a power ( $\lambda$ ) that makes a dataset as close to **normally distributed** as possible. Suppose we have a skewed feature:

$
x = [1, 2, 4, 8, 32]
$

The algorithm tries different values of ( $\lambda$ ) and transforms the data using:

$
y = \frac{x^\lambda - 1}{\lambda}
$

(and when ( $\lambda$ = 0 ), it uses ( y = $\log(x)$ )).

For example:

* **( $\lambda$ = 1 )** → no transformation → still skewed
* **( $\lambda$ = 0.5 )** → square-root–like transformation → values become more compressed
* **( $\lambda$ = 0 )** → log transformation → skewness reduces further

For each candidate ( $\lambda$ ), the transformed data is **assumed to follow a normal distribution**, and the algorithm computes the **log-likelihood** of this assumption using **Maximum Likelihood Estimation (MLE)**. The value of ( $\lambda$ ) that **maximizes this likelihood**—meaning the transformed data best fits a normal distribution—is selected as the optimal parameter. The final dataset is then transformed using this optimal ( $\lambda$ ), resulting in a distribution that is more symmetric and suitable for models that assume Gaussian-like inputs.


# 2. Yeo–Johnson transformer

Think of **Yeo–Johnson** as:

> “Box–Cox, but engineered to work even when data contains zero or negative values.”

Let’s go step-by-step.

---

### Why Yeo–Johnson Exists

The Box–Cox transformation has one big limitation:

$
x > 0
$

It **cannot handle**:

* zeros
* negative numbers

But many real features contain them:

Examples:

* profit/loss
* temperature
* stock returns
* financial ratios
* standardized variables

So statisticians **In-Kwon Yeo and Richard Johnson (2000)** created a generalization.

---

### Core Idea

Yeo–Johnson applies a **power transform similar to Box–Cox**, but uses **different formulas for positive and negative values**.

This keeps the transformation:

* continuous
* differentiable
* symmetric around zero

---

### Yeo–Johnson Formula

For a value (x):

#### If (x $\ge$ 0)

$
y =
\begin{cases}
\frac{(x+1)^\lambda -1}{\lambda}, & \lambda \ne 0 \
\log(x+1), & \lambda = 0
\end{cases}
$

---

#### If (x < 0)

$
y =
\begin{cases}
-\frac{((-x+1)^{2-\lambda}-1)}{2-\lambda}, & \lambda \ne 2 \
-\log(-x+1), & \lambda = 2
\end{cases}
$

The key idea:

* positive values behave like **Box-Cox**
* negative values use a **mirrored transformation**

---

### Example

Suppose we have:

```
x = [-5, -2, -1, 0, 2, 5, 10]
```

Right-skewed data with negatives.

Box-Cox cannot run.

Yeo–Johnson will:

1. Apply one formula to positive values
2. Apply another to negative values
3. Find the **best λ using MLE**
4. Return a distribution closer to **normal**

---

### How λ Is Found

Exactly the same idea as Box-Cox:

1. Try different λ values
2. Transform the data
3. Assume transformed data is normal
4. Compute **log-likelihood**
5. Pick λ that **maximizes likelihood**


So the optimization logic is identical.

---

### What the Transformation Achieves

After Yeo–Johnson:

* skewness reduces
* variance stabilizes
* tails compress
* distribution becomes more symmetric

Which helps models like:

* Linear Regression
* Logistic Regression
* Neural Networks
* PCA
---


# Implementing power transforms 

In [65]:
# without transformation
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler

from sklearn.preprocessing import power_transform


In [66]:
df = pd.read_csv('../assets/data/concrete_data.csv')

In [67]:
df.head()

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
0,540.0,0.0,0.0,162.0,2.5,1040.0,676.0,28,79.99
1,540.0,0.0,0.0,162.0,2.5,1055.0,676.0,28,61.89
2,332.5,142.5,0.0,228.0,0.0,932.0,594.0,270,40.27
3,332.5,142.5,0.0,228.0,0.0,932.0,594.0,365,41.05
4,198.6,132.4,0.0,192.0,0.0,978.4,825.5,360,44.30


In [68]:
df.shape

(1030, 9)

In [69]:
df.isnull().sum() # if any missing values, we will fill it using imputer

Cement                0
Blast Furnace Slag    0
Fly Ash               0
Water                 0
Superplasticizer      0
Coarse Aggregate      0
Fine Aggregate        0
Age                   0
Strength              0
dtype: int64

In [70]:
df.describe() # to check if negative or 0 is present, if present, then we cant apply box-cox

,Cement,Blast Furnace Slag,Fly Ash,Water,Superplasticizer,Coarse Aggregate,Fine Aggregate,Age,Strength
count,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000,1030.000000
mean,281.167864,73.895825,54.188350,181.567282,6.204660,972.918932,773.580485,45.662136,35.817961
std,104.506364,86.279342,63.997004,21.354219,5.973841,77.753954,80.175980,63.169912,16.705742
min,102.000000,0.000000,0.000000,121.800000,0.000000,801.000000,594.000000,1.000000,2.330000
25%,192.375000,0.000000,0.000000,164.900000,0.000000,932.000000,730.950000,7.000000,23.710000
50%,272.900000,22.000000,0.000000,185.000000,6.400000,968.000000,779.500000,28.000000,34.445000
75%,350.000000,142.950000,118.300000,192.000000,10.200000,1029.400000,824.000000,56.000000,46.135000
max,540.000000,359.400000,200.100000,247.000000,32.200000,1145.000000,992.600000,365.000000,82.600000


In [71]:
X = df.drop(columns=["Strength"])
y = df.iloc[:,-1]

In [72]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [75]:
scalar = StandardScaler()

X_train_scaled = scalar.fit_transform(X_train)
X_test_scaled = scalar.transform(X_test)

In [78]:
lr = LinearRegression()

lr.fit(X_train_scaled, y_train)

y_pred = lr.predict(X_test_scaled)

r2_score(y_test, y_pred)

0.627553179231485

In [79]:
lr = LinearRegression()
np.mean(cross_val_score(lr, X, y, scoring='r2'))

np.float64(0.4609940491662864)

In [ ]:
# Plotting the distplots without any transformation
import scipy.stats as stats
for col in X_train.columns:
    plt.figure(figsize=(14,4))
    plt.subplot(121)
    sns.distplot(X_train[col])
    plt.title(col)

    plt.subplot(122)
    stats.probplot(X_train[col], dist="norm", plot=plt)
    plt.title(col)

    plt.show()

In [83]:
pt = PowerTransformer(method='box-cox')

X_train_transformed = pt.fit_transform(X_train+0.00000001) # added 0.00000001 since box-cox doesnt work on 0
X_test_transformed = pt.transform(X_test+0.0000001) 

pd.DataFrame({'cols': X_train.columns, 'box_cox_lambdas': pt.lambdas_})
# these lambdas are used to transform each of the column to normalize the data

,cols,box_cox_lambdas
0,Cement,0.177025
1,Blast Furnace Slag,0.020795
2,Fly Ash,-0.031170
3,Water,0.772682
4,Superplasticizer,0.077874
5,Coarse Aggregate,1.129813
6,Fine Aggregate,1.782018
7,Age,0.066631


In [87]:
lr = LinearRegression()

lr.fit(X_train_transformed, y_train)
y_pred2 = lr.predict(X_test_transformed)

r2_score(y_test, y_pred2)

0.8068697781450452

In [90]:
lr = LinearRegression()

pt = PowerTransformer(method='box-cox')
X_transformed = pt.fit_transform(X+0.0000001)
np.mean(cross_val_score(lr,X_transformed, y, scoring='r2'))

np.float64(0.6658537941434357)